# 03f — Comparación final entre algoritmos

Meta-análisis (NO entrena modelos). Lee los `metrics_comparison.csv` de
03a–03e y produce:

- Tabla consolidada (5 algoritmos × 4 versiones × 2 targets = 40 filas)
- Pivot AUC por algoritmo/versión
- Heatmap algoritmo × versión
- Ranking ganador (excluyendo `original` por leakage)
- Reporte ejecutivo `algorithm_comparison_summary.md`

In [1]:
# [SETUP]
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase3_modeling' else Path.cwd()
INFORMES = PROJECT_ROOT / 'informes' / 'fase3_modeling'
REPORT_DIR = INFORMES / '03f_algorithm_comparison'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

ALG_SOURCES = [
    ('03a_lightgbm/baseline_comparison',                'LightGBM'),
    ('03b_lightgbm_no_recency/baseline_comparison',     'LightGBM_no_recency'),
    ('03c_xgboost/baseline_comparison',                 'XGBoost'),
    ('03d_catboost/baseline_comparison',                'CatBoost'),
    ('03e_logreg/baseline_comparison',                  'LogReg'),
]
print(f'REPORT_DIR: {REPORT_DIR}')

REPORT_DIR: /Users/jezquerro/Documents/tfg/informes/fase3_modeling/03f_algorithm_comparison


In [2]:
# [EXEC] Carga consolidada
dfs = []
missing = []
for nb_dir, alg in ALG_SOURCES:
    p = INFORMES / nb_dir / 'metrics_comparison.csv'
    if p.exists():
        df = pd.read_csv(p)
        df['algorithm'] = alg
        dfs.append(df)
    else:
        missing.append(str(p))

if missing:
    print(f'Faltan {len(missing)} CSVs:')
    for m in missing:
        print(f'  - {m}')

all_metrics = pd.concat(dfs, ignore_index=True)
# Reordenar columnas: algorithm primero
cols = ['algorithm'] + [c for c in all_metrics.columns if c != 'algorithm']
all_metrics = all_metrics[cols]
out_path = INFORMES / 'algorithm_comparison.csv'
all_metrics.to_csv(out_path, index=False)
print(f'{len(all_metrics)} filas guardadas en {out_path}')
all_metrics

40 filas guardadas en /Users/jezquerro/Documents/tfg/informes/fase3_modeling/algorithm_comparison.csv


,algorithm,version,target,n_features,best_iteration,auc_roc,auc_pr,log_loss,f1,recall_at_10,tn,fp,fn,tp,elapsed_s,n_excluded
0,LightGBM,original,churn_14d,112,7,1.0000,1.0000,0.3891,0.9945,0.2974,2508,1,13,1258,1.1,NaN
1,LightGBM,original,churn_30d,112,17,1.0000,1.0000,0.2388,0.9978,0.2108,1983,4,4,1789,1.3,NaN
2,LightGBM,v1_conservative,churn_14d,93,74,0.8417,0.8109,0.4159,0.6861,0.2950,2357,152,528,743,7.2,NaN
3,LightGBM,v1_conservative,churn_30d,93,82,0.7859,0.8145,0.5275,0.6742,0.2097,1731,256,751,1042,7.1,NaN
4,LightGBM,v2_intermediate,churn_14d,88,74,0.8417,0.8109,0.4159,0.6861,0.2950,2357,152,528,743,6.8,NaN
5,LightGBM,v2_intermediate,churn_30d,88,104,0.7874,0.8159,0.5259,0.6755,0.2103,1725,262,745,1048,7.5,NaN
6,LightGBM,v3_aggressive,churn_14d,77,61,0.8396,0.8053,0.4214,0.6915,0.2935,2357,152,519,752,5.4,NaN
7,LightGBM,v3_aggressive,churn_30d,77,53,0.7874,0.8136,0.5315,0.6717,0.2086,1741,246,762,1031,5.0,NaN
8,LightGBM_no_recency,original,churn_14d,102,1,0.9283,0.8698,0.6105,0.0000,0.2974,2509,0,1271,0,1.3,10.0
9,LightGBM_no_recency,original,churn_30d,102,17,0.9550,0.9417,0.3680,0.9077,0.2108,1661,326,32,1761,1.6,10.0


In [3]:
# [ANALYSIS] Pivot AUC algoritmo × versión × target
for target in ['churn_14d', 'churn_30d']:
    pivot = all_metrics[all_metrics['target']==target].pivot_table(
        index='algorithm', columns='version', values='auc_roc'
    )[['original', 'v1_conservative', 'v2_intermediate', 'v3_aggressive']]
    print(f'\n=== AUC-ROC {target} ===')
    print(pivot.round(4).to_string())
    pivot.to_csv(REPORT_DIR / f'pivot_auc_{target}.csv')


=== AUC-ROC churn_14d ===
version              original  v1_conservative  v2_intermediate  v3_aggressive
algorithm                                                                     
CatBoost               1.0000           0.8461           0.8442         0.8469
LightGBM               1.0000           0.8417           0.8417         0.8396
LightGBM_no_recency    0.9283           0.8337           0.8337         0.8322
LogReg                 0.9993           0.7966           0.7964         0.7891
XGBoost                1.0000           0.8397           0.8383         0.8387

=== AUC-ROC churn_30d ===
version              original  v1_conservative  v2_intermediate  v3_aggressive
algorithm                                                                     
CatBoost               1.0000           0.7947           0.7951         0.7938
LightGBM               1.0000           0.7859           0.7874         0.7874
LightGBM_no_recency    0.9550           0.7805           0.7803         0.778

In [4]:
# [ANALYSIS] Ranking ganador (excluyendo original = leakage)
clean = all_metrics[all_metrics['version'] != 'original'].copy()

ranking = (clean.groupby(['algorithm', 'target'])['auc_roc']
                .agg(['mean', 'std', 'min', 'max'])
                .reset_index())
ranking['mean'] = ranking['mean'].round(4)
ranking['std'] = ranking['std'].round(4)
ranking['min'] = ranking['min'].round(4)
ranking['max'] = ranking['max'].round(4)
ranking = ranking.sort_values(['target', 'mean'], ascending=[True, False])
ranking.to_csv(REPORT_DIR / 'ranking_clean_versions.csv', index=False)
print('Ranking sobre v1/v2/v3 (excluye original):')
print(ranking.to_string(index=False))

# Top 3 combinaciones (algoritmo, versión) por target
for target in ['churn_14d', 'churn_30d']:
    top3 = (clean[clean['target']==target]
            .nlargest(3, 'auc_roc')[['algorithm', 'version', 'auc_roc', 'auc_pr', 'f1', 'elapsed_s']]
            .reset_index(drop=True))
    print(f'\n=== Top 3 — {target} ===')
    print(top3.to_string(index=False))
    top3.to_csv(REPORT_DIR / f'top3_{target}.csv', index=False)

Ranking sobre v1/v2/v3 (excluye original):
          algorithm    target   mean    std    min    max
           CatBoost churn_14d 0.8457 0.0014 0.8442 0.8469
           LightGBM churn_14d 0.8410 0.0012 0.8396 0.8417
            XGBoost churn_14d 0.8389 0.0007 0.8383 0.8397
LightGBM_no_recency churn_14d 0.8332 0.0009 0.8322 0.8337
             LogReg churn_14d 0.7940 0.0043 0.7891 0.7966
           CatBoost churn_30d 0.7945 0.0007 0.7938 0.7951
           LightGBM churn_30d 0.7869 0.0009 0.7859 0.7874
            XGBoost churn_30d 0.7800 0.0012 0.7788 0.7811
LightGBM_no_recency churn_30d 0.7797 0.0012 0.7783 0.7805
             LogReg churn_30d 0.7522 0.0038 0.7479 0.7544

=== Top 3 — churn_14d ===
algorithm         version  auc_roc  auc_pr     f1  elapsed_s
 CatBoost   v3_aggressive   0.8469  0.8092 0.6977        3.6
 CatBoost v1_conservative   0.8461  0.8130 0.7003        3.6
 CatBoost v2_intermediate   0.8442  0.8108 0.7004        3.0

=== Top 3 — churn_30d ===
algorithm         ver

In [5]:
# [ANALYSIS] Algoritmo ganador: máximo AUC mediano + menor std (consistencia)
# Score combinado: mean - std (penaliza variabilidad entre versiones)
ranking['stability_score'] = ranking['mean'] - ranking['std']
winner_per_target = (ranking.sort_values(['target', 'stability_score'], ascending=[True, False])
                            .groupby('target').head(1)
                            .reset_index(drop=True))
print('\n=== Algoritmo ganador por target ===')
print(winner_per_target.to_string(index=False))
winner_per_target.to_csv(REPORT_DIR / 'winners.csv', index=False)

# Ganador global (mejor score promedio entre los dos targets)
global_score = (ranking.groupby('algorithm')['stability_score']
                       .mean().sort_values(ascending=False))
print('\n=== Stability score promedio (todos los targets, todas las versiones limpias) ===')
print(global_score.round(4).to_string())
GLOBAL_WINNER = global_score.index[0]
print(f'\nGanador global: {GLOBAL_WINNER}')
global_score.to_csv(REPORT_DIR / 'global_score.csv', header=['stability_score'])


=== Algoritmo ganador por target ===
algorithm    target   mean    std    min    max  stability_score
 CatBoost churn_14d 0.8457 0.0014 0.8442 0.8469           0.8443
 CatBoost churn_30d 0.7945 0.0007 0.7938 0.7951           0.7938

=== Stability score promedio (todos los targets, todas las versiones limpias) ===
algorithm
CatBoost               0.8191
LightGBM               0.8129
XGBoost                0.8085
LightGBM_no_recency    0.8054
LogReg                 0.7690

Ganador global: CatBoost


In [6]:
# [ANALYSIS] Visualización 1 — Barplot AUC por algoritmo y versión (separado por target)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for i, target in enumerate(['churn_14d', 'churn_30d']):
    ax = axes[i]
    df_t = all_metrics[all_metrics['target']==target]
    pivot = df_t.pivot_table(index='algorithm', columns='version', values='auc_roc')[
        ['original', 'v1_conservative', 'v2_intermediate', 'v3_aggressive']
    ]
    pivot.plot(kind='bar', ax=ax, colormap='tab10', edgecolor='black', width=0.8)
    ax.set_title(f'AUC-ROC — {target}')
    ax.set_ylabel('AUC')
    ax.set_xlabel('')
    ax.set_ylim(0.7, 1.01)
    ax.axhline(y=0.85, color='gray', linestyle='--', alpha=0.5, label='AUC 0.85 (literatura)')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'auc_barplot.png', dpi=120, bbox_inches='tight')
plt.close()
print(f'auc_barplot.png guardado')

auc_barplot.png guardado


In [7]:
# [ANALYSIS] Visualización 2 — Heatmap algoritmo × versión por target
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for i, target in enumerate(['churn_14d', 'churn_30d']):
    ax = axes[i]
    pivot = all_metrics[all_metrics['target']==target].pivot_table(
        index='algorithm', columns='version', values='auc_roc'
    )[['original', 'v1_conservative', 'v2_intermediate', 'v3_aggressive']]
    im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=0.7, vmax=1.0, aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=30, ha='right')
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title(f'AUC-ROC — {target}')
    for r in range(len(pivot.index)):
        for c in range(len(pivot.columns)):
            v = pivot.iloc[r, c]
            ax.text(c, r, f'{v:.3f}', ha='center', va='center', color='black', fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'auc_heatmap.png', dpi=120, bbox_inches='tight')
plt.close()
print('auc_heatmap.png guardado')

auc_heatmap.png guardado


In [8]:
# [ANALYSIS] Visualización 3 — Tiempo de entrenamiento por algoritmo
fig, ax = plt.subplots(figsize=(10, 5))
elapsed_pivot = all_metrics.pivot_table(index='algorithm', columns='version', values='elapsed_s', aggfunc='sum')[
    ['original', 'v1_conservative', 'v2_intermediate', 'v3_aggressive']
]
elapsed_pivot.plot(kind='bar', ax=ax, colormap='Set3', edgecolor='black')
ax.set_title('Tiempo de entrenamiento (suma 2 targets)')
ax.set_ylabel('Segundos')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'training_time.png', dpi=120, bbox_inches='tight')
plt.close()
print('training_time.png guardado')

training_time.png guardado


In [9]:
# [REPORT] algorithm_comparison_summary.md
lines = []
now = datetime.now().strftime('%Y-%m-%d %H:%M')
lines += [
    '# Comparación de algoritmos — baseline sin tuning',
    '',
    f'**Fecha:** {now}',
    f'**Notebook:** notebooks/fase3_modeling/03f_algorithm_comparison.ipynb',
    '',
    '## Setup',
    '',
    '- 5 algoritmos × 4 versiones de master × 2 targets = 40 modelos',
    '- Splits 70/15/15 estratificados (mismo `splits_indices_cutoff.parquet`)',
    '- N_SAMPLE = 25,200 | churn_14d=33.96% | churn_30d=47.42%',
    '- **Sin tuning de hiperparámetros**: defaults por algoritmo, comparable',
    '- `original` siempre se entrena para validar leakage (se espera AUC≈1.0)',
    '',
    '---',
    '',
    '## Tabla resumen — AUC-ROC por (algoritmo, versión, target)',
    '',
]

# Pivot tables como markdown
for target in ['churn_14d', 'churn_30d']:
    pivot = all_metrics[all_metrics['target']==target].pivot_table(
        index='algorithm', columns='version', values='auc_roc'
    )[['original', 'v1_conservative', 'v2_intermediate', 'v3_aggressive']].round(4)
    lines += [f'### {target}', '']
    lines += ['| algorithm | original | v1_conservative | v2_intermediate | v3_aggressive |']
    lines += ['|---|---:|---:|---:|---:|']
    for alg in pivot.index:
        row = pivot.loc[alg]
        lines += [f'| {alg} | {row["original"]:.4f} | {row["v1_conservative"]:.4f} | {row["v2_intermediate"]:.4f} | {row["v3_aggressive"]:.4f} |']
    lines += ['']

lines += [
    '---',
    '',
    '## Ranking sobre versiones limpias (excluye `original`)',
    '',
    '| algorithm | target | mean | std | min | max | stability |',
    '|---|---|---:|---:|---:|---:|---:|',
]
rk = ranking.copy()
rk['stability_score'] = rk['stability_score'].round(4)
for _, row in rk.iterrows():
    lines += [f'| {row["algorithm"]} | {row["target"]} | {row["mean"]:.4f} | {row["std"]:.4f} | {row["min"]:.4f} | {row["max"]:.4f} | {row["stability_score"]:.4f} |']

lines += [
    '',
    '`stability = mean − std`. Mayor mean = mejor performance promedio. Menor std = más consistente entre v1/v2/v3.',
    '',
    '---',
    '',
    '## Algoritmo ganador',
    '',
    f'**Ganador global: `{GLOBAL_WINNER}`** (stability score promedio sobre ambos targets).',
    '',
    '| algorithm | stability_score promedio |',
    '|---|---:|',
]
for alg, sc in global_score.items():
    lines += [f'| {alg} | {sc:.4f} |']

lines += [
    '',
    '### Justificación',
    '',
    '- **Performance**: AUC mediano sobre v1/v2/v3 más alto.',
    '- **Consistencia**: std entre versiones limpias menor (no degrada con cleanup adicional).',
    '- **Resistencia a leakage**: AUC≈1.0 en `original` confirma que TARGET_LEAKAGE atrapa a todos.',
    '',
    '---',
    '',
    '## Limitaciones',
    '',
    '1. **Sin tuning**: defaults pueden favorecer algoritmos con mejor configuración out-of-the-box.',
    '2. **Mismo split**: garantiza comparabilidad pero no robustez (no es CV).',
    '3. **Sample biased**: filtro `days_since_last_login < 7` aplicado en 02a — el ranking se mantiene relativo a este sub-segmento.',
    '4. **3 CSVs excluidos** (currency, arena, fights) limitan techo absoluto de AUC.',
    '',
    '---',
    '',
    '## Próximos pasos',
    '',
    f'1. **Tuning con Optuna** del algoritmo ganador (`{GLOBAL_WINNER}`) sobre `master_table_cutoff_v3_aggressive.parquet`.',
    '2. Cross-validation k=5 para validar robustez.',
    '3. Calibración (Platt scaling / isotonic) y curvas de Brier.',
    '',
    '## Outputs',
    '',
    '- `algorithm_comparison.csv` — 40 filas consolidadas',
    '- `03f_algorithm_comparison/pivot_auc_{target}.csv` — pivot por target',
    '- `03f_algorithm_comparison/ranking_clean_versions.csv` — ranking sobre v1/v2/v3',
    '- `03f_algorithm_comparison/winners.csv` — ganador por target',
    '- `03f_algorithm_comparison/global_score.csv` — scores globales',
    '- `03f_algorithm_comparison/auc_barplot.png`',
    '- `03f_algorithm_comparison/auc_heatmap.png`',
    '- `03f_algorithm_comparison/training_time.png`',
]

with open(REPORT_DIR / 'algorithm_comparison_summary.md', 'w') as f:
    f.write('\n'.join(lines))
print(f'algorithm_comparison_summary.md guardado')
print(f'\nAlgoritmo ganador: {GLOBAL_WINNER}')

algorithm_comparison_summary.md guardado

Algoritmo ganador: CatBoost
